# Prepare and Plot NASS Milk Production Data

This notebook analyzes USDA NASS monthly and annual milk production data downloaded from NASS Quick Stats.

## Mount Google Drive in Colab

In [ ]:
# Uncomment if using in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

## Data Loading

Import libraries and load the USDA NASS milk production dataset from the project data directory.

In [ ]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

# Import visualization libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Uncomment if using in Google Colab
# milk = pd.read_csv('drive/MyDrive/AGEC-370/Data/milk_production_lbs_nass.csv')
milk = pd.read_csv('data/milk_production_1924_2026_nass.csv')

### Inspect Raw Data

Preview the first few rows and column types to understand the raw structure before cleaning.

In [ ]:
milk.head()

In [ ]:
milk.info()

## Data Cleaning

The `Value` column from NASS data often contains suppressed entries marked `(D)` or `(Z)`, comma-formatted numbers, and mixed types. The steps below standardize it into a clean numeric column.

In [ ]:
value_clean = milk['Value'].fillna('').astype(str)

# Remove any row containing (D) or (Z) anywhere
mask = value_clean.str.contains(r'\([DZ]\)', regex=True)
milk = milk[~mask].copy()

# Remove ',' from `Value` column
milk['Value'] = (
    milk['Value']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
)
# Convert to numeric
milk['Value'] = pd.to_numeric(milk['Value'], errors='coerce')

# Check for NAs
print(milk['Value'].isna().sum())

# Optional: Drop NAs if needed
# milk.dropna(subset=['Value'], inplace=True) 

In [ ]:
# Keep only columns needed for analysis; drops NASS metadata not used downstream
sel_columns = ['Year', 'Period', 'Data Item', 'Value']
milk = milk[sel_columns]

In [ ]:
# Standardize to snake_case for consistent attribute access throughout the notebook
rename_cols = milk.columns.str.lower().str.replace(' ', '_')
milk.columns = rename_cols

In [ ]:
# Rename to a domain-specific label so column intent is clear downstream
milk = milk.rename(columns={'value': 'prod_lbs'})

In [ ]:
# NASS reports production in pounds; dividing by 1B converts to billion lbs for readable axis labels
milk['prod_lbs'] = milk['prod_lbs'] / 1_000_000_000

In [ ]:
# Remove quarterly aggregates - they overlap with individual months and would double-count production
quarterly_periods = ['JAN THRU MAR', 'APR THRU JUN', 'JUL THRU SEP', 'OCT THRU DEC']
milk = milk[~milk['period'].isin(quarterly_periods)].copy()

# Define chronological period order: YEAR first, then Jan through Dec
month_order = ['YEAR', 'JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN',
               'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

# Cast period as ordered categorical so pandas sorts by calendar order, not alphabetically
milk['period'] = pd.Categorical(milk['period'], categories=month_order, ordered=True)

# Sort: newest year first; within each year, YEAR row first then Jan through Dec
milk = milk.sort_values(['year', 'period'], ascending=[False, True]).reset_index(drop=True)

milk.head(20)

In [ ]:
# Separate annual totals for use in the annual trend plot
milk_yearly = milk[milk['period'] == 'YEAR']

## Data Visualization

We plot milk production at two levels: (1) annual totals to show the long-run trend, and (2) monthly values to reveal seasonality and shorter-term patterns.

In [ ]:
sns.set_style('whitegrid')  # clean background with subtle gridlines; avoids busy chart decoration

### Annual Milk Production - Line Plot

Documentation: [Seaborn Line Plot](https://seaborn.pydata.org/generated/seaborn.lineplot.html)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(14, 6))  # wide canvas suits a century-long time series

sns.lineplot(
    data=milk_yearly,
    x='year',
    y='prod_lbs',
    linewidth=1.0,
    color='darkblue',  # dark blue contrasts cleanly against the white grid background
    ax=ax
)

ax.set_title("Annual Milk Production (Billion lbs)", fontsize=14, weight='bold')
ax.set_xlabel("Year", fontsize=12, weight='bold')
ax.set_ylabel("Production (Billion lbs)", fontsize=12, weight='bold')

# Tick every 3 years prevents overlapping labels on a dense x-axis spanning 100+ years
all_years = sorted(milk_yearly['year'].unique())
ax.set_xticks(all_years[::3])
ax.set_xticklabels(all_years[::3])
ax.tick_params(axis='x', rotation=90)

plt.tight_layout()

# Save before plt.show() - calling show() clears the figure from memory
# plt.savefig('/content/drive/MyDrive/AGEC-370/Figures/annual_milk_production.png', bbox_inches='tight')
plt.savefig('output/figures/milk_production_annual.png', bbox_inches='tight')

plt.show()

### Average Monthly Production by Month

Aggregate all years by month to reveal the seasonal production cycle — which months peak and which lag.

In [ ]:
# Filter to monthly rows only - exclude annual YEAR totals
milk_monthly = milk[milk['period'] != 'YEAR']

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(14, 6))

# estimator='mean' aggregates all years per month, showing the average seasonal pattern across the full dataset
sns.lineplot(
    data=milk_monthly,
    x='period',
    y='prod_lbs',
    estimator='mean',
    color='darkblue',
    linewidth=1.0,
    ax=ax
)

ax.set_title("Average Monthly Milk Production (Billion lbs)", fontsize=14, weight='bold')
ax.set_xlabel("Month", fontsize=12, weight='bold')
ax.set_ylabel("Production (Billion lbs)", fontsize=12, weight='bold')

plt.tight_layout()

# Save before plt.show() - calling show() clears the figure from memory
# plt.savefig('/content/drive/MyDrive/AGEC-370/Figures/monthly_milk_production_avg.png', bbox_inches='tight')
plt.savefig('output/figures/milk_production_monthly_avg.png', bbox_inches='tight')

plt.show()

### Monthly Production Time Series (2010–2024)

Plot month-by-month production as a continuous time series. A proper `datetime` column is constructed so matplotlib can handle chronological ordering and x-axis tick spacing automatically.

In [ ]:
# Filter to monthly rows only - exclude annual YEAR totals
milk_monthly = milk[milk['period'] != 'YEAR'].copy()

milk_monthly = milk_monthly[milk_monthly['year'] >= 2010]

# Map abbreviated month names to integers for datetime construction
month_map = {'JAN':1, 'FEB':2, 'MAR':3, 'APR':4,  'MAY':5,  'JUN':6,
             'JUL':7, 'AUG':8, 'SEP':9, 'OCT':10, 'NOV':11, 'DEC':12}

# Build a proper datetime column; day=1 is a placeholder since data is monthly frequency
milk_monthly['date'] = pd.to_datetime({
    'year':  milk_monthly['year'],
    'month': milk_monthly['period'].map(month_map),
    'day':   1
})

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

sns.lineplot(data=milk_monthly, x='date', y='prod_lbs', ax=ax,
             color='steelblue', linewidth=0.8)  # thin line suits dense monthly data

# Tick every 2 years keeps the x-axis readable without crowding
ax.xaxis.set_major_locator(mdates.YearLocator(2))

ax.set_title('U.S. Monthly Milk Production (2010-2024)', fontsize=14, weight='bold', pad=12)
ax.set_xlabel('Year', fontsize=11, weight='bold')
ax.set_ylabel('Production (Billion lbs)', fontsize=11, weight='bold')

plt.tight_layout()

# Save before plt.show() - calling show() clears the figure from memory
# plt.savefig('/content/drive/MyDrive/AGEC-370/Figures/monthly_milk_production_2010_2024.png', bbox_inches='tight')
plt.savefig('output/figures/milk_production_monthly_2010_2024.png', bbox_inches='tight')

plt.show()

### Questions

Which months show the highest milk production, and which show the lowest? Why might this seasonal pattern exist?